# Frosty Friday Week 104 — 4解法の比較

元課題: https://www.frostyfri.day/en/challenges/blog/2024/08/02/week-104-cortex-ai-/-ml

PDF → テキスト抽出 → チャンキング → 要約のパイプラインを **4つの解法** で実装する。

| 解法 | 抽出 | 要約 | 出力テーブル |
|---|---|---|---|
| ① Legacy (2024) | PyPDF2 (Python UDTF) | `SUMMARIZE` | `STORIES_SUMMARY_LEGACY` |
| ② AISQL (2026) | `AI_PARSE_DOCUMENT` | `AI_SUMMARIZE_AGG` | `STORIES_SUMMARY_AISQL` |
| ③ Extract | `AI_PARSE_DOCUMENT` | `AI_EXTRACT` | `STORIES_STRUCTURED_TABLE` |
| ④ Complete (DIY) | `AI_PARSE_DOCUMENT` | `AI_COMPLETE` + プロンプト | `STORIES_SUMMARY_COMPLETE` |

## 比較の2軸

- **縦軸 (時系列)**: ① 2024年のPython実装 vs ②③④ 2026年のSQL完結
- **横軸 (責任分界)**: ②③ マネージド型 (プロンプト不要・様式と品質を関数側が担保)
  vs ④ DIY型 (粒度・言語をプロンプトで指示でき、その保守責任を利用者が持つ)

## 妥当性確認 (Phase 7)

4解法の要約を日本語化して目視確認。品質差の判定はしない (n=3 では誤差と区別不能)。

## 実行前提

- 実行ロールはNotebook設定で指定 (`USE ROLE` はNotebook内で実行不可)
- 既存データベース `FROSTY_FRIDAY` 配下に `week_104` スキーマを作成して進める (DB自体は作成しない)
- SQLセルは1セル1ステートメント
- 増分パイプライン構成 (Stream + Task DAG) は別添の完全版SQLを参照


## Phase 1: スキーマ / ウェアハウス

既存DB `FROSTY_FRIDAY` 配下に `week_104` スキーマを作成する。
DBは他Weekのスキーマと共有のため、作成も削除もしない。

In [ ]:
CREATE SCHEMA IF NOT EXISTS FROSTY_FRIDAY.week_104;

In [ ]:
USE SCHEMA FROSTY_FRIDAY.week_104;

In [ ]:
-- AISQLの推論はCortex側コンピュートで実行されるため、ローカルWHは最小でよい
CREATE WAREHOUSE IF NOT EXISTS FROSTY_WH
  WAREHOUSE_SIZE      = 'XSMALL'
  AUTO_SUSPEND        = 60
  AUTO_RESUME         = TRUE
  INITIALLY_SUSPENDED = FALSE;

In [ ]:
USE WAREHOUSE FROSTY_WH;

## Phase 2: ステージ作成と PDF アップロード

ステージはアップロード済みファイルを保持するため `IF NOT EXISTS` で作成する
(`CREATE OR REPLACE` は再実行時にファイルを消す)。

### アップロード手順 (Snowsight UI操作)

1. 元ストーリーPDF (3つ) を取得: https://github.com/psophe/Frosty_Friday_Stories_w104
2. Snowsight → Data → Databases → `FROSTY_FRIDAY` → `WEEK_104` → Stages → `STORIES_STAGE`
3. **+ Files** から3つのPDFをアップロード
4. 次のREFRESHセルを実行 (内部ステージは自動リフレッシュされない)


In [ ]:
-- ステージはファイルを保持するため IF NOT EXISTS (OR REPLACE はファイルを消す)
-- AI_PARSE_DOCUMENT は SNOWFLAKE_SSE 暗号化のステージのみ対応
CREATE STAGE IF NOT EXISTS STORIES_STAGE
  DIRECTORY  = (ENABLE = TRUE)
  ENCRYPTION = (TYPE   = 'SNOWFLAKE_SSE');

In [ ]:
ALTER STAGE STORIES_STAGE REFRESH;

In [ ]:
-- ★エビデンス: 3行返ること (0行ならアップロードまたはREFRESH漏れ)
SELECT relative_path, size, last_modified
FROM DIRECTORY(@STORIES_STAGE);

## Phase 3: 解法① Legacy — Python UDTF + SUMMARIZE (2024年公式想定解)

PDF抽出を Python UDTF (PyPDF2 + LangChain) で行い、`SUMMARIZE` で要約。
出力は比較用に `STORIES_SUMMARY_LEGACY` へ。


In [ ]:
-- 使用パッケージがAnacondaチャネルに存在することの確認
SELECT package_name, version
FROM INFORMATION_SCHEMA.PACKAGES
WHERE language = 'python'
  AND package_name IN ('pypdf2', 'langchain-text-splitters',
                       'snowflake-snowpark-python')
QUALIFY ROW_NUMBER() OVER (PARTITION BY package_name ORDER BY version DESC) = 1;

In [ ]:
CREATE OR REPLACE FUNCTION pdf_text_chunker_legacy(file_url STRING)
RETURNS TABLE (chunk VARCHAR)
LANGUAGE PYTHON
RUNTIME_VERSION = '3.10'
HANDLER  = 'pdf_text_chunker'
PACKAGES = ('snowflake-snowpark-python', 'pypdf2', 'langchain-text-splitters')
AS
$$
from snowflake.snowpark.files import SnowflakeFile
from PyPDF2 import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import io

class pdf_text_chunker:
    def read_pdf(self, file_url):
        # 通常の open() ではなく SnowflakeFile.open() を使う
        with SnowflakeFile.open(file_url, 'rb') as f:
            buffer = io.BytesIO(f.readall())
        reader = PdfReader(buffer)
        text = ""
        for page in reader.pages:
            # extract_text() は None を返し得るため空文字フォールバック
            # \0 (NULL文字) は VARCHAR で例外を起こすので除去
            page_text = page.extract_text() or ""
            text += page_text.replace('\n', ' ').replace('\0', ' ')
        return text

    def process(self, file_url: str):
        text = self.read_pdf(file_url)
        splitter = RecursiveCharacterTextSplitter(
            chunk_size      = 4000,
            chunk_overlap   = 400,
            length_function = len
        )
        for chunk in splitter.split_text(text):
            yield (chunk,)
$$;

In [ ]:
CREATE OR REPLACE TABLE STORIES_CHUNKS_LEGACY AS
SELECT
    d.relative_path AS story_title,
    f.chunk
FROM DIRECTORY(@STORIES_STAGE) d,
     LATERAL pdf_text_chunker_legacy(
         BUILD_SCOPED_FILE_URL(@STORIES_STAGE, d.relative_path)
     ) f;

In [ ]:
-- ストーリー単位で全チャンク連結 → SUMMARIZE
CREATE OR REPLACE TABLE STORIES_SUMMARY_LEGACY AS
SELECT
    story_title,
    SNOWFLAKE.CORTEX.SUMMARIZE(
        LISTAGG(chunk, ' ') WITHIN GROUP (ORDER BY chunk)
    ) AS summary
FROM STORIES_CHUNKS_LEGACY
GROUP BY story_title;

In [ ]:
-- ★エビデンス: 解法① 完了 (3行)
SELECT * FROM STORIES_SUMMARY_LEGACY ORDER BY story_title;

## Phase 4: 解法② AISQL — AI_PARSE_DOCUMENT + AI_SUMMARIZE_AGG (2026年版)

パース結果は `STORIES_PARSED` に永続化し、解法③とも共有する
(`AI_PARSE_DOCUMENT` の実行は1回で済ませ、クレジットを節約)。

実装メモ: `AI_PARSE_DOCUMENT` の戻り値は OBJECT 型のため `:content` で直接パスアクセスする。
パース失敗時はエラーではなく NULL が返るため `IS NOT NULL` で明示的に除外する。


In [ ]:
-- 共通パース層: AI_PARSE_DOCUMENT を1回だけ実行して永続化
CREATE OR REPLACE TABLE STORIES_PARSED AS
SELECT
    d.relative_path AS story_title,
    AI_PARSE_DOCUMENT(
        TO_FILE('@STORIES_STAGE', d.relative_path),
        {'mode': 'LAYOUT'}
    ):content::STRING AS full_text
FROM DIRECTORY(@STORIES_STAGE) d;

In [ ]:
-- ★エビデンス: 3行・full_textがNULLでないこと
SELECT story_title, LENGTH(full_text) AS text_length
FROM STORIES_PARSED;

In [ ]:
CREATE OR REPLACE TABLE STORIES_SUMMARY_AISQL AS
WITH chunked AS (
    SELECT
        story_title,
        c.value::STRING AS chunk
    FROM STORIES_PARSED,
         LATERAL FLATTEN(
             SNOWFLAKE.CORTEX.SPLIT_TEXT_RECURSIVE_CHARACTER(
                 full_text,
                 'markdown',     -- LAYOUTモードの出力 (Markdown) に合わせる
                 4000,
                 400
             )
         ) c
    WHERE full_text IS NOT NULL
)
SELECT
    story_title,
    AI_SUMMARIZE_AGG(chunk) AS summary
FROM chunked
GROUP BY story_title;

In [ ]:
-- ★エビデンス: 解法② 完了 (3行)
SELECT * FROM STORIES_SUMMARY_AISQL ORDER BY story_title;

## Phase 5: 解法③ Extract — AI_EXTRACT による要約+構造化

`AI_EXTRACT` はプロンプトで複数フィールドを一括抽出でき、`scores => TRUE` で
フィールドごとの信頼度スコアを返す (実構造: `scoring:scores:<field>:score`)。

**検証済みの役割分担 (n=3)**:
- 固有エンティティ抽出 (main_character) → 3/3正解・スコア0.89〜0.96で高位安定。**EXTRACTの適所**
- 解釈を要するフィールド (theme / ending) → スコアは0.22〜0.73でばらつき、
  **原文由来の回答でも低スコアになり得る**。スコアは正答指標ではなく、
  「低スコア行を人間レビューに回す」トリアージ用途に限る
- 存在しない情報の検出は response の "None" / NULL を主条件にする (スコア閾値は補助)
- `summary` (生成タスク) → 抜粋⇔生成が非一貫。**要約には不採用**


In [ ]:
CREATE OR REPLACE TABLE STORIES_STRUCTURED_TABLE AS
SELECT
    story_title,
    AI_EXTRACT(
        text           => full_text,
        responseFormat => {
            'summary':        'Summarize this story in 3 sentences.',
            'main_character': 'Who is the main character?',
            'theme':          'What is the central theme of the story?',
            'ending':         'How does the story end? Answer in one sentence.'
        },
        scores         => TRUE
    ) AS extracted
FROM STORIES_PARSED
WHERE full_text IS NOT NULL;

In [ ]:
-- ★エビデンス: 解法③ 完了
-- scores の実構造は extracted:scoring:scores:<field>:score (実出力で確認済み)
-- スコアは well-defined な抽出 (main_character) で高位安定、解釈タスク (theme/ending) で
-- ばらつく。閾値による自動採否ではなく、低スコア行のトリアージに使う
SELECT
    story_title,
    extracted:response:summary::STRING                AS summary,
    extracted:scoring:scores:summary:score::FLOAT     AS summary_score,
    extracted:response:main_character::STRING         AS main_character,
    extracted:scoring:scores:main_character:score::FLOAT AS mc_score,
    extracted:response:theme::STRING                  AS theme,
    extracted:scoring:scores:theme:score::FLOAT       AS theme_score
FROM STORIES_STRUCTURED_TABLE
ORDER BY story_title;

## Phase 6: 解法④ Complete — 汎用モデル + 自前プロンプト (DIY型)

②③のマネージド関数に対し、④は `AI_COMPLETE` に**プロンプトで粒度を指示する**方式。
今回の粒度 (3文程度の要約) はプロンプトで指定し、`temperature 0` で再現性を固定する。

プレーンテキスト出力のため様式リスクは小さい (JSON等の**構造化出力を要求する場合は**
フェンス混入・キー脱落が起き得るため、防御パースとリトライをセットで持つ)。
プロンプトは書いた瞬間から組織の保守資産になる — それがこの方式のコスト。


In [ ]:
CREATE OR REPLACE TABLE STORIES_SUMMARY_COMPLETE AS
SELECT
    story_title,
    AI_COMPLETE(
        'claude-4-sonnet',
        CONCAT(
            'Summarize the following story in about 3 sentences. ',
            'Return only the summary text.\n\n',
            full_text
        ),
        {'temperature': 0}      -- 再現性の固定
    ) AS summary
FROM STORIES_PARSED
WHERE full_text IS NOT NULL;

In [ ]:
-- ★エビデンス: 解法④ 完了 (3行)
SELECT * FROM STORIES_SUMMARY_COMPLETE ORDER BY story_title;

## Phase 7: 妥当性確認 (日本語)

4解法の要約を日本語で読み、物語の要約として妥当かを確認する。
品質差の判定はしない (n=3 では誤差と区別できない)。
明らかな破綻 (原文と無関係な内容、文章の体を成していない等) がないことだけを見る。


In [ ]:
-- ★エビデンス: 4解法の要約を日本語で妥当性確認
SELECT '1_legacy' AS method, story_title, summary,
       AI_TRANSLATE(summary, 'en', 'ja') AS summary_ja
FROM STORIES_SUMMARY_LEGACY
UNION ALL
SELECT '2_aisql', story_title, summary,
       AI_TRANSLATE(summary, 'en', 'ja')
FROM STORIES_SUMMARY_AISQL
UNION ALL
SELECT '3_extract', story_title,
       extracted:response:summary::STRING,
       AI_TRANSLATE(extracted:response:summary::STRING, 'en', 'ja')
FROM STORIES_STRUCTURED_TABLE
UNION ALL
SELECT '4_complete', story_title, summary,
       AI_TRANSLATE(summary, 'en', 'ja')
FROM STORIES_SUMMARY_COMPLETE
ORDER BY story_title, method;

## Phase 8: 最終成果物の確定とコスト確認

妥当性確認を踏まえ、チャレンジ要件の `STORIES_SUMMARY_TABLE` を確定する。
`CORTEX_FUNCTIONS_USAGE_HISTORY` は反映に2〜3時間ラグがあるため、
実行直後は0行になり得る。


In [ ]:
-- チャレンジ要件の最終テーブルを確定
-- 採用: 解法② (AISQL)
-- 根拠は工学的性質: プロンプト資産の保守が不要で、様式・品質をマネージド側が担保。
-- ④COMPLETEは同等の出力を出せるが、プロンプトが組織の保守対象になる。
-- 使い分け: 定型タスクは専用関数、粒度・言語等の細かい制御が要件のときにCOMPLETE。
-- 要約品質は n=3 では解法間の差を誤差と区別できず、目視でも実用上同等。
-- (確定的な差は様式のみ: 解法③のsummaryは抜粋が混ざるため要約には不採用)
CREATE OR REPLACE TABLE STORIES_SUMMARY_TABLE AS
SELECT story_title, summary FROM STORIES_SUMMARY_AISQL;

In [ ]:
-- ★エビデンス: チャレンジ要件 (3行)
SELECT COUNT(*) AS row_count FROM STORIES_SUMMARY_TABLE;

In [ ]:
-- ★エビデンス: Cortex関数別のクレジット消費 (直近1日)
SELECT
    function_name,
    model_name,
    SUM(token_credits) AS total_token_credits,
    COUNT(*)           AS call_count
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_FUNCTIONS_USAGE_HISTORY
WHERE start_time >= DATEADD('day', -1, CURRENT_TIMESTAMP())
GROUP BY function_name, model_name
ORDER BY total_token_credits DESC;

## 後片付け (検証完了後のみ・誤実行防止のためコードブロックで記載)

```sql
DROP TABLE  IF EXISTS STORIES_STRUCTURED_TABLE;
DROP TABLE  IF EXISTS STORIES_SUMMARY_COMPLETE;
DROP TABLE  IF EXISTS STORIES_SUMMARY_AISQL;
DROP TABLE  IF EXISTS STORIES_SUMMARY_LEGACY;
DROP TABLE  IF EXISTS STORIES_SUMMARY_TABLE;
DROP TABLE  IF EXISTS STORIES_PARSED;
DROP TABLE  IF EXISTS STORIES_CHUNKS_LEGACY;
DROP FUNCTION IF EXISTS pdf_text_chunker_legacy(STRING);
DROP STAGE  IF EXISTS STORIES_STAGE;     -- 中のPDFも消える点に注意
DROP SCHEMA IF EXISTS FROSTY_FRIDAY.week_104;
-- DB FROSTY_FRIDAY は他Weekと共有のため削除しない
```
